## Cell 1 — Imports & Logging

In [0]:
import os
import io
import logging
import zipfile
import hashlib
import requests
import pandas as pd

from datetime import datetime, timezone
from pyspark.sql import SparkSession, DataFrame
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StringType, DoubleType, IntegerType, BooleanType
)

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(name)s — %(message)s"
)
logger = logging.getLogger("stl_gtfs_ingest")

print("✓ Imports loaded")

## Cell 2 — Configuration

In [0]:
# ── Output paths ──────────────────────────────────────────────
# GTFS static data lands here as Parquet, partitioned by
# ingestion date so we can track schedule changes over time.
RAW_OUTPUT_PATH = "/Volumes/workspace/default/raw/gtfs"

# Checkpoint file stores the MD5 hash of the last successfully
# processed ZIP. On each scheduled run we compare the new
# download's hash against this value — if they match, the
# schedule hasn't changed and we skip reprocessing.
# Stored as a simple text file alongside the Parquet output.
CHECKPOINT_PATH = f"{RAW_OUTPUT_PATH}/_checkpoint/last_zip_hash.txt"

# ── Source URL ────────────────────────────────────────────────
# Confirmed directly from metrostlouis.org/developer-resources/
# Metro updates this file when schedule changes occur, roughly
# 5 times per year. No date-based URL logic needed — it's
# always the same URL pointing to the current schedule.
GTFS_ZIP_URL = "https://www.metrostlouis.org/Transit/google_transit.zip"

# ── Files we extract from the ZIP ────────────────────────────
# A GTFS ZIP contains many files — we only need three for the
# neighborhood intelligence model:
#   stops.txt  — lat/lon of every bus stop and MetroLink station
#   routes.txt — route names, colors, and types (bus vs rail)
#   trips.txt  — which trips run on which routes and shapes
# Other files (calendar.txt, stop_times.txt, shapes.txt) are
# available but not needed at this stage.
GTFS_FILES = ["stops.txt", "routes.txt", "trips.txt"]

# ── HTTP headers ──────────────────────────────────────────────
# Same browser-spoofing headers we use across all notebooks
# to avoid 403 blocks on government/agency web servers.
BROWSER_HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/124.0.0.0 Safari/537.36"
    ),
    "Accept": "application/zip, application/octet-stream, */*",
}

print("✓ Configuration set")
print(f"  Source URL:      {GTFS_ZIP_URL}")
print(f"  Output path:     {RAW_OUTPUT_PATH}")
print(f"  Checkpoint path: {CHECKPOINT_PATH}")
print(f"  Files to extract: {GTFS_FILES}")

## Cell 3 — Checkpoint Helpers

In [0]:
def read_checkpoint(path: str) -> str | None:
    """
    Read the MD5 hash stored from the last successful ingestion run.

    The checkpoint is a plain text file containing a single MD5
    hash string. If the file doesn't exist yet (first run) or
    can't be read, we return None which tells the caller to
    proceed with full ingestion.

    Args:
        path: DBFS or Volume path to the checkpoint text file.

    Returns:
        MD5 hash string from last run, or None if not found.
    """
    try:
        # dbutils.fs.head reads the first N bytes of a file.
        # 32 bytes is exactly the length of an MD5 hex digest.
        content = dbutils.fs.head(path, 32)
        if content:
            logger.info(f"  Checkpoint found: {content.strip()}")
            return content.strip()
    except Exception:
        # File doesn't exist yet — this is expected on first run
        logger.info("  No checkpoint found — first run or checkpoint missing")
    return None


def write_checkpoint(path: str, md5_hash: str) -> None:
    """
    Write the MD5 hash of the successfully processed ZIP to the
    checkpoint file for use on the next scheduled run.

    Creates parent directories if they don't exist.

    Args:
        path:     DBFS or Volume path to write the checkpoint to.
        md5_hash: MD5 hex digest string to store.
    """
    try:
        # Ensure the checkpoint directory exists
        parent = "/".join(path.split("/")[:-1])
        dbutils.fs.mkdirs(parent)

        # Write hash as plain text — dbutils.fs.put overwrites
        # any existing file at this path
        dbutils.fs.put(path, md5_hash, overwrite=True)
        logger.info(f"  Checkpoint written: {md5_hash}")
    except Exception as e:
        # Non-fatal — log but don't crash the pipeline.
        # Next run will just reprocess rather than skip.
        logger.warning(f"  Could not write checkpoint: {e}")


def compute_md5(data: bytes) -> str:
    """
    Compute the MD5 hex digest of a byte string.

    Used to fingerprint the downloaded ZIP file so we can detect
    whether Metro has published a new schedule since the last run.

    Args:
        data: Raw bytes to hash.

    Returns:
        32-character lowercase hex string, e.g. "d41d8cd98f00b204..."
    """
    return hashlib.md5(data).hexdigest()


print("✓ Checkpoint helpers defined")

## Cell 4 — Download Helper

In [0]:
def download_gtfs_zip(url: str) -> bytes | None:
    """
    Download the Metro STL GTFS static ZIP file.

    Retries up to 3 times on transient failures. Returns raw
    bytes on success or None if all attempts fail.

    The timeout is set higher than our other notebooks (90s)
    because the GTFS ZIP can be several MB and the Metro web
    server is occasionally slow to respond.

    Args:
        url: Full URL to the GTFS ZIP file.

    Returns:
        Raw ZIP bytes, or None on failure.
    """
    for attempt in range(1, 4):
        try:
            logger.info(f"  GET {url} (attempt {attempt}/3)")
            resp = requests.get(
                url,
                headers=BROWSER_HEADERS,
                timeout=90   # higher timeout — ZIP can be several MB
            )

            if resp.status_code == 200:
                size_mb = len(resp.content) / 1_000_000
                logger.info(f"  ✓ Downloaded {size_mb:.2f} MB")
                return resp.content

            # 403 — server blocked the request, no point retrying
            if resp.status_code == 403:
                logger.error(f"  403 Forbidden — check browser headers")
                return None

            # 404 — URL has changed, needs investigation
            if resp.status_code == 404:
                logger.error(f"  404 Not Found — URL may have changed: {url}")
                return None

            # Any other status — log and retry
            logger.warning(f"  HTTP {resp.status_code} on attempt {attempt}/3")

        except requests.Timeout:
            logger.warning(f"  Timeout on attempt {attempt}/3")
        except requests.RequestException as e:
            logger.warning(f"  Request error on attempt {attempt}/3: {e}")

    logger.error("  All 3 download attempts failed")
    return None


print("✓ download_gtfs_zip() defined")

## Cell 5 — GTFS File Schemas

In [0]:
# Expected columns for each GTFS file we extract.
# These come directly from the GTFS specification:
# https://developers.google.com/transit/gtfs/reference
#
# We define required vs optional separately:
#   REQUIRED — must be present or we raise an error
#   OPTIONAL — included if present, null-filled if missing
#
# This matters because Metro may not populate every optional
# field, and we don't want ingestion to fail on missing columns.

GTFS_SCHEMAS = {

    "stops.txt": {
        # Each row is one physical stop — a bus stop pole or
        # MetroLink station. stop_lat/stop_lon place it on the map.
        "required": [
            "stop_id",       # unique identifier, e.g. "1234"
            "stop_name",     # human-readable name, e.g. "Grand & Gravois"
            "stop_lat",      # latitude (WGS84 decimal degrees)
            "stop_lon",      # longitude (WGS84 decimal degrees)
        ],
        "optional": [
            "stop_code",     # short public-facing code shown at stop
            "stop_desc",     # longer description
            "zone_id",       # fare zone identifier
            "stop_url",      # URL of stop-specific webpage
            "location_type", # 0=stop, 1=station, 2=entrance
            "parent_station",# stop_id of parent station (e.g. MetroLink)
            "wheelchair_boarding", # 0=unknown, 1=accessible, 2=not
        ],
    },

    "routes.txt": {
        # Each row is one route — a named service like "40 Gravois"
        # or the Red MetroLink line.
        "required": [
            "route_id",      # unique identifier
            "route_short_name", # short public name, e.g. "40"
            "route_long_name",  # full name, e.g. "Gravois-Hampton"
            "route_type",    # 0=tram, 1=subway, 2=rail, 3=bus
        ],
        "optional": [
            "agency_id",     # operator identifier
            "route_desc",    # longer description
            "route_url",     # URL for route info page
            "route_color",   # hex color for map display
            "route_text_color", # hex color for text on route color
        ],
    },

    "trips.txt": {
        # Each row is one trip — a single vehicle journey along
        # a route at a specific time. Links routes to shapes.
        "required": [
            "route_id",      # foreign key → routes.txt
            "service_id",    # links to calendar.txt for schedule
            "trip_id",       # unique identifier for this trip
        ],
        "optional": [
            "trip_headsign",  # destination text shown on vehicle
            "trip_short_name",# public trip identifier if any
            "direction_id",   # 0=outbound, 1=inbound
            "block_id",       # vehicle block (for scheduling)
            "shape_id",       # foreign key → shapes.txt
            "wheelchair_accessible", # 0=unknown, 1=yes, 2=no
            "bikes_allowed",  # 0=unknown, 1=yes, 2=no
        ],
    },
}

print("✓ GTFS schemas defined")
for filename, schema in GTFS_SCHEMAS.items():
    total = len(schema["required"]) + len(schema["optional"])
    print(f"  {filename}: {len(schema['required'])} required + "
          f"{len(schema['optional'])} optional = {total} total columns")

## Cell 6 — Parse Single GTFS File

In [0]:
def parse_gtfs_file(
    zip_bytes: bytes,
    filename: str,
) -> pd.DataFrame | None:
    """
    Extract and parse one file from the GTFS ZIP.

    GTFS files are plain CSVs inside a ZIP archive. This function:
      1. Opens the ZIP from bytes (no temp file needed)
      2. Reads the target .txt file as CSV
      3. Validates required columns are present
      4. Adds null columns for any missing optional fields
      5. Drops any extra columns not in our schema
      6. Attaches provenance metadata

    Args:
        zip_bytes: Raw bytes of the downloaded GTFS ZIP.
        filename:  Name of the file to extract, e.g. "stops.txt"

    Returns:
        Pandas DataFrame or None if extraction fails.
    """
    # Retrieve the schema definition for this file
    schema = GTFS_SCHEMAS.get(filename)
    if schema is None:
        logger.error(f"  No schema defined for {filename}")
        return None

    all_cols     = schema["required"] + schema["optional"]
    required_cols = schema["required"]

    # ── Step 1: Open ZIP from bytes ───────────────────────────
    # io.BytesIO lets us treat the raw bytes as a file-like object
    # without writing anything to disk. Cleaner and faster.
    try:
        with zipfile.ZipFile(io.BytesIO(zip_bytes)) as zf:

            # List available files for debugging — helps if
            # Metro renames a file in a future GTFS update
            available = zf.namelist()
            logger.info(f"  ZIP contains: {available}")

            if filename not in available:
                logger.error(
                    f"  {filename} not found in ZIP. "
                    f"Available files: {available}"
                )
                return None

            # ── Step 2: Read the CSV file ─────────────────────
            # GTFS spec requires UTF-8 encoding with optional BOM.
            # encoding="utf-8-sig" handles both cases — it strips
            # the BOM if present, otherwise reads as plain UTF-8.
            with zf.open(filename) as f:
                pdf = pd.read_csv(
                    f,
                    dtype=str,          # keep all values as strings
                    encoding="utf-8-sig", # handles optional BOM
                    low_memory=False
                )

    except zipfile.BadZipFile as e:
        logger.error(f"  Invalid ZIP file: {e}")
        return None
    except Exception as e:
        logger.error(f"  Error reading {filename}: {e}")
        return None

    # Strip whitespace from column names — GTFS files sometimes
    # have trailing spaces in headers
    pdf.columns = [c.strip() for c in pdf.columns]

    logger.info(
        f"  {filename}: {len(pdf):,} rows, "
        f"{len(pdf.columns)} columns"
    )

    # ── Step 3: Validate required columns ────────────────────
    missing_required = [
        c for c in required_cols
        if c not in pdf.columns
    ]
    if missing_required:
        logger.error(
            f"  {filename} missing required columns: "
            f"{missing_required}"
        )
        return None

    # ── Step 4: Add missing optional columns as null ─────────
    # Guarantees output always has all expected columns even if
    # Metro didn't populate an optional field in this release.
    for col in schema["optional"]:
        if col not in pdf.columns:
            pdf[col] = None

    # ── Step 5: Keep only schema columns ─────────────────────
    # Drops any GTFS fields we don't need to keep the output
    # schema clean and predictable.
    pdf = pdf[[c for c in all_cols if c in pdf.columns]]

    # ── Step 6: Attach provenance metadata ───────────────────
    pdf["_source"]      = f"metro_stl_gtfs/{filename}"
    pdf["_ingested_at"] = datetime.now(timezone.utc).isoformat()

    logger.info(f"  ✓ Parsed {filename}: {len(pdf):,} rows")
    return pdf


print("✓ parse_gtfs_file() defined")

## Cell 7 — Convert to Spark & Write Parquet

In [0]:
def write_gtfs_table(
    spark: SparkSession,
    pdf: pd.DataFrame,
    filename: str,
    output_base: str,
    ingest_date: str,
) -> int:
    """
    Convert a parsed GTFS Pandas DataFrame to Spark and write
    it to Parquet, partitioned by ingest_date.

    We partition by ingest_date (not data date, since GTFS is
    a static snapshot) so we can track how the schedule changes
    over time across scheduled runs. This lets us answer questions
    like "when did route 40 stop serving this neighborhood?"

    Args:
        spark:       Active SparkSession.
        pdf:         Parsed Pandas DataFrame from parse_gtfs_file().
        filename:    Source filename e.g. "stops.txt" — used to
                     name the output subdirectory.
        output_base: Base Parquet output path.
        ingest_date: ISO date string "YYYY-MM-DD" for partition.

    Returns:
        Row count written.
    """
    # Derive the table name from the filename, e.g.
    # "stops.txt" → "stops"
    table_name = filename.replace(".txt", "")

    # Build the full output path including partition
    output_path = (
        f"{output_base}/{table_name}"
        f"/ingest_date={ingest_date}"
    )

    # Convert Pandas → Spark
    sdf = spark.createDataFrame(pdf)

    # Cast every column to StringType before writing.
    # Columns that are entirely null get inferred as VOID type
    # by Spark, which Parquet cannot store. Explicitly casting to
    # string prevents this — GTFS fields are all strings anyway,
    # and nulls survive the cast as null strings.
    sdf = sdf.select([
        F.col(c).cast(StringType()).alias(c)
        for c in sdf.columns
    ])

    # Write with overwrite — safe to re-run on the same date
    sdf.write.mode("overwrite").parquet(output_path)

    row_count = sdf.count()
    logger.info(
        f"  ✓ Wrote {row_count:,} rows → {output_path}"
    )
    return row_count


print("✓ write_gtfs_table() defined")

## Cell 8 — Main Ingestion Orchestrator

In [0]:
def run_gtfs_ingestion(
    spark: SparkSession,
    force: bool = False,
) -> dict[str, int]:
    """
    Main entry point for GTFS static ingestion.

    Downloads the Metro STL GTFS ZIP, checks whether it has
    changed since the last run using MD5 hashing, and if so
    extracts and writes stops, routes, and trips to Parquet.

    This function is designed to be called on a schedule (e.g.
    weekly Databricks Job). On most runs the ZIP will be
    identical to last time and the function will exit early,
    making scheduled runs cheap and idempotent.

    Args:
        spark: Active SparkSession.
        force: If True, skip the hash check and always reprocess.
               Useful for manual re-runs or debugging.

    Returns:
        Dict mapping table name → row count written.
        Empty dict if ingestion was skipped (no change detected).
    """
    ingest_date = datetime.now(timezone.utc).strftime("%Y-%m-%d")
    results: dict[str, int] = {}

    print(f"GTFS ingestion starting — {ingest_date}")
    print("=" * 60)

    # ── Step 1: Download the ZIP ──────────────────────────────
    logger.info(f"Downloading GTFS ZIP from {GTFS_ZIP_URL}")
    zip_bytes = download_gtfs_zip(GTFS_ZIP_URL)

    if zip_bytes is None:
        raise RuntimeError(
            "GTFS ZIP download failed. "
            "Check network access and verify the URL is still valid."
        )

    # ── Step 2: Hash check — skip if unchanged ────────────────
    # Compute MD5 of the downloaded file and compare to the
    # hash stored from the last successful run.
    current_hash  = compute_md5(zip_bytes)
    previous_hash = read_checkpoint(CHECKPOINT_PATH)

    logger.info(f"Current ZIP hash:  {current_hash}")
    logger.info(f"Previous ZIP hash: {previous_hash or 'none'}")

    if not force and current_hash == previous_hash:
        # Hash matches — Metro hasn't published a new schedule.
        # Exit early to avoid unnecessary reprocessing.
        print(
            f"\n⏭ ZIP unchanged since last run (hash: {current_hash})\n"
            f"  Skipping reprocessing. Use force=True to override."
        )
        return {}

    if previous_hash is None:
        print("\n📥 First run — no previous checkpoint found")
    else:
        print(f"\n🔄 ZIP changed — new schedule detected")
        print(f"  Previous hash: {previous_hash}")
        print(f"  Current hash:  {current_hash}")

    # ── Step 3: Extract and write each GTFS file ──────────────
    for filename in GTFS_FILES:
        print(f"\n{'─' * 40}")
        print(f"Processing {filename}...")

        pdf = parse_gtfs_file(zip_bytes, filename)

        if pdf is None:
            print(f"  ⚠ Skipped {filename} — parse failed")
            continue

        row_count = write_gtfs_table(
            spark,
            pdf,
            filename,
            RAW_OUTPUT_PATH,
            ingest_date,
        )

        table_name          = filename.replace(".txt", "")
        results[table_name] = row_count
        print(f"  ✓ {filename}: {row_count:,} rows written")

    # ── Step 4: Save checkpoint ───────────────────────────────
    # Only write the checkpoint after ALL files have been
    # successfully processed. If any file failed, we want the
    # next run to reprocess rather than think this run succeeded.
    if len(results) == len(GTFS_FILES):
        write_checkpoint(CHECKPOINT_PATH, current_hash)
        print(f"\n✓ Checkpoint updated: {current_hash}")
    else:
        print(
            f"\n⚠ Not all files processed successfully "
            f"({len(results)}/{len(GTFS_FILES)}) — "
            f"checkpoint NOT updated"
        )

    # ── Summary ───────────────────────────────────────────────
    print(f"\n{'=' * 60}")
    print(f"GTFS ingestion complete — {ingest_date}")
    for table, count in results.items():
        print(f"  {table:10s}: {count:,} rows")

    return results


print("✓ run_gtfs_ingestion() defined")

## Cell 9 — Run Ingestion

In [0]:
# Set force=True on first run or to reprocess regardless of
# whether the ZIP has changed. Set force=False (default) for
# scheduled runs so unchanged feeds are skipped automatically.
results = run_gtfs_ingestion(spark, force=False)

## Cell 10 — Quality Checks

In [0]:
if not results:
    print("Ingestion was skipped — ZIP unchanged. "
          "Run with force=True to inspect data.")
else:
    # ── Stops ────────────────────────────────────────────────
    print("=== STOPS — NULL RATES (%) ===")
    stops_df = spark.read.parquet(f"{RAW_OUTPUT_PATH}/stops")
    display(stops_df.select([
        F.round(
            F.count(F.when(F.col(c).isNull(), c)) /
            F.count(F.lit(1)) * 100, 2
        ).alias(c)
        for c in GTFS_SCHEMAS["stops.txt"]["required"]
    ]))

    # COMMAND ----------

    print("=== STOPS — LOCATION SAMPLE (confirm lat/lon look right) ===")
    display(
        stops_df.select("stop_id", "stop_name", "stop_lat", "stop_lon")
        .orderBy("stop_name")
        .limit(10)
    )

    # COMMAND ----------

    print("=== ROUTES — TYPE DISTRIBUTION ===")
    routes_df = spark.read.parquet(f"{RAW_OUTPUT_PATH}/routes")
    # route_type: 0=tram, 1=subway/MetroLink, 3=bus
    display(
        routes_df.groupBy("route_type")
        .count()
        .orderBy("route_type")
    )

    # COMMAND ----------

    print("=== TRIPS — COUNT BY ROUTE (Top 20) ===")
    trips_df = spark.read.parquet(f"{RAW_OUTPUT_PATH}/trips")
    display(
        trips_df.groupBy("route_id")
        .count()
        .orderBy("count", ascending=False)
        .limit(20)
    )

## Cell 11 — Verify Parquet Write

In [0]:
print("=== PARQUET VERIFICATION ===")
for filename in GTFS_FILES:
    table_name = filename.replace(".txt", "")
    path       = f"{RAW_OUTPUT_PATH}/{table_name}"
    try:
        df    = spark.read.parquet(path)
        count = df.count()
        print(f"  ✓ {table_name:10s}: {count:,} rows at {path}")
    except Exception as e:
        print(f"  ✗ {table_name:10s}: Could not read — {e}")

## Cell 12 — Run Unit Tests



In [0]:
import subprocess
import sys
import os
import shutil
import tempfile

# ── Install pytest ────────────────────────────────────────────
subprocess.run(
    [sys.executable, "-m", "pip", "install", "pytest", "--quiet"],
    check=True
)

# ── Repo root ─────────────────────────────────────────────────
REPO_ROOT      = "/Workspace/Users/jcoffey@wustl.edu/Data_Engineering_Final"
TEST_FILE_PATH = os.path.join(REPO_ROOT, "tests", "test_ingest_gtfs.py")

if not os.path.exists(TEST_FILE_PATH):
    raise FileNotFoundError(f"Test file not found at {TEST_FILE_PATH}")

# ── Copy test file to a writable temp directory ───────────────
# The Databricks Workspace filesystem doesn't support __pycache__
# writes that pytest requires. Copying to /tmp which is a normal
# writable filesystem sidesteps this entirely.
tmpdir = tempfile.mkdtemp()
tmp_test_file = os.path.join(tmpdir, "test_ingest_gtfs.py")
shutil.copy2(TEST_FILE_PATH, tmp_test_file)
print(f"Copied test file to: {tmp_test_file}")

# ── Add repo root to Python path ──────────────────────────────
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

# ── Run pytest from temp directory ───────────────────────────
result = subprocess.run(
    [
        sys.executable, "-m", "pytest",
        tmp_test_file,
        "-v",
        "--tb=short",
        "--no-header",
    ],
    capture_output=True,
    text=True,
    cwd=tmpdir,
    env={**os.environ, "PYTHONPATH": REPO_ROOT},
)

# ── Cleanup temp directory ────────────────────────────────────
shutil.rmtree(tmpdir, ignore_errors=True)

print(result.stdout)
if result.stderr:
    print("STDERR:")
    print(result.stderr)

if result.returncode == 0:
    print("\n✓ All tests passed")
else:
    print(f"\n✗ Tests failed (exit code {result.returncode})")
    raise RuntimeError("Test suite failed — check output above")